# Lesson 2.9 — Expert demonstrations with a motion planner

Everything up to here used **random** actions. This notebook produces what the project
never had before: trajectories that actually **succeed**, and can therefore serve as
expert demonstrations for imitation learning.

The planner is ManiSkill's own sampling-based motion planner, the same component its
built-in demonstration generation uses. The recipe is taken from the reference
implementation shipped with the package:

```text
mani_skill/examples/motionplanning/panda/solutions/pick_cube.py
```

## Two results that shape this notebook

**1. The planner needed a NumPy downgrade.** `mplib` 0.1.1 — which ManiSkill 3.0.1 pins
with `Requires-Dist: mplib==0.1.1` — is compiled against the NumPy 1.x C API. Under
NumPy 2.x its stored function pointers are invalid, so constructing the planner jumps to
address `0x0` and the process dies with `SIGSEGV`:

```text
Fatal Python error: Segmentation fault
  File ".../mplib/planner.py", line 65 in __init__
  File ".../base_motionplanner/motionplanner.py", line 59 in setup_planner
  File ".../panda/motionplanner.py", line 25 in __init__
```

Kernel log: `segfault at 0 ip 0000000000000000`. The fix is `numpy<2`, **not** a GPU or
backend setting: the same script segfaults under NumPy 2.2.6 and succeeds under
NumPy 1.26.4, with CUDA unavailable in both cases.

**2. Control mode and the gripper steps are load-bearing.** The planner's
`close_gripper()` / `open_gripper()` emit `[qpos(7), gripper]`, so the environment must
run `pd_joint_pos`. And the official recipe **does not open the gripper** after carrying
the cube to the goal — adding `open_gripper()` plus settling steps drops the cube beside
the goal and the episode fails with `is_obj_placed: False`.

This notebook records what actually works.

## 2.9.1 — Preamble

In [1]:
import gymnasium as gym
import mani_skill.envs
import numpy as np
import torch

torch.set_printoptions(precision=4, sci_mode=False)


def make_env(obs_mode="state", control_mode="pd_joint_pos", seed=0):
    env = gym.make(
        "PickCube-v1",
        obs_mode=obs_mode,
        control_mode=control_mode,
        num_envs=1,
    )
    env.reset(seed=seed)
    return env

## Environment separation

This lesson uses two environments:

| Environment | Purpose |
|---|---|
| embodied | general learning, VLA experiments |
| embodied310 | ManiSkill expert generation |

The expert-generation environment uses:
- Python 3.10
- NumPy 1.26.4
- mplib 0.1.1

because mplib 0.1.1 is not compatible with NumPy 2.x.

## 2.9.2 — Why the planner cannot run in this notebook's environment

The project has two Python environments, and they differ exactly on the library this
notebook needs:

| Environment | NumPy | mplib planner |
|---|---|---|
| `embodied` (notebook default) | `2.2.6` | **segfaults** |
| `embodied310` (planning) | `1.26.4` | works |

So this notebook **does not construct the planner**. Doing so kills the kernel outright —
a segmentation fault cannot be caught by `try` / `except`, and there is no error to
record, only a dead process.

The planner work is delegated to `scripts/generate_expert_demo.py`, run with
`embodied310`. What can be inspected safely here is the **action contract** the planner
depends on, which is the real reason the control mode cannot be `pd_joint_delta_pos`.

In [2]:
import gymnasium as gym
import mani_skill.envs

# The planner's close_gripper() / open_gripper() step the env with
# action = [qpos(7), gripper]  -- an (8,) ABSOLUTE joint position vector.
# Compare the action space of the two candidate control modes.
for mode in ("pd_joint_pos", "pd_joint_delta_pos"):
    e = gym.make("PickCube-v1", obs_mode="state", control_mode=mode, num_envs=1)
    space = e.action_space
    normalized = bool(np.allclose(space.low, -1.0) and np.allclose(space.high, 1.0))
    print(f"{mode:<20} dim={space.shape[0]}  normalized={normalized}")
    if not normalized:
        print(f"{'':<20} low  = {np.round(space.low, 4)}")
        print(f"{'':<20} high = {np.round(space.high, 4)}")
    e.close()

print()
print("pd_joint_pos       : absolute joint targets inside the joint limits.")
print("                     [qpos(7), gripper] fits directly -- this is what the planner emits.")
print(
    "pd_joint_delta_pos : normalized deltas + absolute gripper; "
    "feeding a position vector directly is incorrect."
)
print("                     into it fails with 'Received action of shape torch.Size([15])'.")

2026-09-22 11:37:37,858 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1061: UserWarning: Can't initialize NVML
  raw_cnt = _raw_device_count_nvml()
/home/bowenyuan/miniforge3/envs/embodied/lib/python3.12/site-packages/torch/cuda/__init__.py:1113: UserWarning: CUDA initialization: Unexpected error from cudaGetDeviceCount(). Did you run some cuda functions before calling NumCudaDevices() that might have already set an error? Error 304: OS call failed or operation not supported on this OS (Triggered internally at /pytorch/c10/cuda/CUDAFunctions.cpp:119.)
  r = torch._C._cuda_getDeviceCount() if nvml_count < 0 else nvml_count


2026-09-22 11:37:38,280 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


pd_joint_pos         dim=8  normalized=False
                     low  = [-2.8973 -1.7628 -2.8973 -3.0718 -2.8973 -0.0175 -2.8973 -1.    ]
                     high = [ 2.8973  1.7628  2.8973 -0.0698  2.8973  3.7525  2.8973  1.    ]
pd_joint_delta_pos   dim=8  normalized=True

pd_joint_pos       : absolute joint targets inside the joint limits.
                     [qpos(7), gripper] fits directly -- this is what the planner emits.
pd_joint_delta_pos : normalized deltas + absolute gripper; feeding a position vector directly is incorrect.
                     into it fails with 'Received action of shape torch.Size([15])'.


In [3]:
# ==================================================
# Demonstrate action semantic mismatch
#
# Motion planner generates:
#     absolute joint positions
#
#     [q1, q2, ..., q7, gripper]
#
# However:
#
# pd_joint_delta_pos expects:
#     joint position deltas
#
# They may have the same dimension,
# but represent different meanings.
# ==================================================


env_delta = gym.make(
    "PickCube-v1",
    obs_mode="state",
    control_mode="pd_joint_delta_pos",
    num_envs=1
)


obs, info = env_delta.reset(seed=0)


print("Delta controller action space:")
print(env_delta.action_space)


# --------------------------------------------------
# A valid delta action
#
# Meaning:
#     move joints slightly from current position
# --------------------------------------------------

delta_action = np.zeros(
    env_delta.action_space.shape,
    dtype=np.float32
)


obs, reward, terminated, truncated, info = env_delta.step(
    delta_action
)


print("-----------------------------")
print("Delta action accepted")
print("Reward:", reward)


env_delta.close()

2026-09-22 11:37:38,417 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


Delta controller action space:
Box(-1.0, 1.0, (8,), float32)
-----------------------------
Delta action accepted
Reward: tensor([0.0653])


## Important: Action Semantics

`pd_joint_delta_pos` and `pd_joint_pos` may have the same action dimension,
but they represent different control commands.

---

### `pd_joint_pos`

**Action:**

```text
[q1, q2, ..., q7, gripper]
```

**Meaning:**

Move the robot to these target joint positions.

The action represents the **absolute target joint configuration**.

---

### `pd_joint_delta_pos`

**Action:**

```text
[Δq1, Δq2, ..., Δq7, gripper]
```

**Meaning:**

Change the current joint positions by these increments.

The action represents the **relative joint movement** from the current state.

---

### Expert Demonstration

The `mplib` motion planner generates absolute joint trajectories:

```text
current robot state
        |
        v
   mplib planner
        |
        v
[q1, q2, ..., q7, gripper]
```

Therefore, expert demonstrations use:

```text
pd_joint_pos
```

rather than:

```text
pd_joint_delta_pos
```

### Why the planner bypasses the action space

`move_to_pose_with_screw` plans a joint-space motion and drives the robot's joint targets
directly. It bypasses `env.step(action)` for the arm motion, then uses `env.step` for the
gripper and the settling steps.

That matters for data collection: a planner produces **trajectories**, and turning a
trajectory into training data requires the actions the environment actually executed. The
collector script wraps `env.step` so every transition is recorded as it happens, rather
than being reconstructed afterwards — see section 2.9.6.

## 2.9.3 — The grasping geometry

A grasp pose needs three things, and none of them is a magic number:

- **approaching**: the direction the gripper comes from. Straight down, because the cube
  sits on a table.
- **closing**: the axis the fingers close along, derived from the object's oriented
  bounding box. This is what makes the grasp work for a cube at an arbitrary yaw.
- **center**: where to grasp.

`target_closing` is taken from the TCP's own rotation: the arm's current finger axis is
reused instead of inventing one. `build_grasp_pose` returns a **SAPIEN** `Pose`, not a
ManiSkill `Pose` — so it exposes `.p` and `.q` directly, not `.raw_pose`.

In [4]:
import sapien

from mani_skill.examples.motionplanning.base_motionplanner.utils import (
    compute_grasp_info_by_obb,
    get_actor_obb,
)

FINGER_LENGTH = 0.025

env = make_env(obs_mode="state", control_mode="pd_joint_pos", seed=0)
unwrapped = env.unwrapped

obb = get_actor_obb(unwrapped.cube)

approaching = np.array([0, 0, -1])
# y axis of the TCP frame = the finger closing axis
target_closing = unwrapped.agent.tcp.pose.to_transformation_matrix()[0, :3, 1].cpu().numpy()

grasp_info = compute_grasp_info_by_obb(
    obb,
    approaching=approaching,
    target_closing=target_closing,
    depth=FINGER_LENGTH,
)
closing = grasp_info["closing"]
center = grasp_info["center"]
grasp_pose = unwrapped.agent.build_grasp_pose(approaching, closing, unwrapped.cube.pose.sp.p)

print("cube position :", unwrapped.cube.pose.sp.p)
print("goal position :", unwrapped.goal_site.pose.sp.p)
print("approaching   :", approaching)
print("closing axis  :", np.round(closing, 4))
print("grasp center  :", np.round(center, 4))
print("grasp pose type:", type(grasp_pose).__name__, "(SAPIEN Pose)")
print("grasp pose p  :", np.round(grasp_pose.p, 4))
print("grasp pose q  :", np.round(grasp_pose.q, 4))

# Reach pose: grasp pose offset 5 cm upward, so the gripper descends from above
reach_pose = grasp_pose * sapien.Pose([0, 0, -0.05])
print("reach  pose p :", np.round(reach_pose.p, 4))

# Goal pose reuses the grasp orientation so the cube stays upright on the way over
goal_pose = sapien.Pose(unwrapped.goal_site.pose.sp.p, grasp_pose.q)
print("goal   pose p :", np.round(goal_pose.p, 4))

2026-09-22 11:37:38,563 - mani_skill  - WARNING - Requested to use render device "sapien_cuda", but CUDA device was not found. Falling back to "cpu" device. Rendering might be disabled.


cube position : [-0.00074868  0.05364437  0.02      ]
goal position : [ 0.02681573 -0.00198132  0.28893346]
approaching   : [ 0  0 -1]
closing axis  : [ 0.353  -0.9356  0.    ]
grasp center  : [-0.0007  0.0536  0.02  ]
grasp pose type: Pose (SAPIEN Pose)
grasp pose p  : [-0.0007  0.0536  0.02  ]
grasp pose q  : [0.     0.9838 0.1794 0.    ]
reach  pose p : [-0.0007  0.0536  0.07  ]
goal   pose p : [ 0.0268 -0.002   0.2889]


## 2.9.4 — Execute the plan

The verified sequence is **four steps, and no more**:

```python
planner.move_to_pose_with_screw(reach_pose)   # approach from above
planner.move_to_pose_with_screw(grasp_pose)   # descend
planner.close_gripper()                       # grasp
planner.move_to_pose_with_screw(goal_pose)    # carry to the goal
```

Two negative results are worth stating, because both look reasonable and both fail:

- **`open_gripper()` after the carry** — the cube is released while it is still beside
  the goal, fails the placement distance check, and `is_obj_placed` comes back `False`.
- **Extra zero-action settling steps** — they do not fix the placement; they let the cube
  drift further.

Note that `close_gripper()` and `open_gripper()` internally call `env.step` with an
`(8,)` action of the form `[qpos(7), gripper]`. That is why the environment must be
`pd_joint_pos`: under `pd_joint_delta_pos` the helper produces a 15-d vector and the
controller rejects it with
`AssertionError: Received action of shape torch.Size([15]) but expected shape (1, 8)`.

In [5]:
import subprocess
import sys
from pathlib import Path

# The planner must run in embodied310 (numpy<2), so invoke the collector as a
# subprocess rather than constructing the planner in this kernel.
PLANNING_PYTHON = Path.home() / "miniforge3" / "envs" / "embodied310" / "bin" / "python"
from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
COLLECTOR = PROJECT_ROOT / "scripts" / "generate_expert_demo.py"

print("planning interpreter:", PLANNING_PYTHON, "exists:", PLANNING_PYTHON.exists())

if PLANNING_PYTHON.exists():
    result = subprocess.run(
        [str(PLANNING_PYTHON), str(COLLECTOR), "--seeds", "0", "1", "2", "3", "4", "--overwrite"],
        capture_output=True, text=True, cwd=str(PROJECT_ROOT),
    )
    print("exit code:", result.returncode)
    print(result.stdout.strip()[-1200:])
    if result.returncode != 0:
        print("STDERR:", result.stderr.strip()[-600:])
else:
    print("embodied310 not found; run the collector manually:")
    print("  python scripts/generate_expert_demo.py --seeds 0 1 2 3 4 --overwrite")

planning interpreter: /home/bowenyuan/miniforge3/envs/embodied310/bin/python exists: True


exit code: 0
environment : PickCube-v1
control mode: pd_joint_pos
seeds       : [0, 1, 2, 3, 4]
seed 0: frames=74 success=True placed=True static=True return=24.9710
seed 1: frames=74 success=True placed=True static=True return=23.1076
seed 2: frames=50 success=True placed=True static=True return=14.7990
seed 3: frames=86 success=True placed=True static=True return=27.8694
seed 4: frames=76 success=True placed=True static=True return=25.6839

successful episodes: 5/5
written: /home/bowenyuan/Projects/embodied-ai-learning/datasets/pickcube/expert_episodes.h5

NOTE: this is expert data in pd_joint_pos semantics. The project's existing fixture is pd_joint_delta_pos; see notes/progress.md for how the two relate.


## 2.9.5 — What success means

Success is a property of the **task evaluation**, not of the planner returning without
error:

```python
info = env.unwrapped.evaluate()
# {"success", "is_obj_placed", "is_robot_static", "is_grasped"}
```

| Field | Meaning |
|---|---|
| `is_obj_placed` | cube within `goal_thresh` (0.025 m) of the goal |
| `is_robot_static` | arm has settled |
| `success` | `is_obj_placed AND is_robot_static` |

**Grasping is not success.** An arm holding the cube in mid-air has not completed the
task. That is why "the trajectory looked reasonable" is not an acceptance criterion
anywhere in this project.

In [6]:
print("goal threshold:", unwrapped.goal_thresh, "m")
print("cube half size:", unwrapped.cube_half_size)
print()
print("evaluate() keys:", list(unwrapped.evaluate().keys()))
print("success requires BOTH is_obj_placed AND is_robot_static.")

goal threshold: 0.025 m
cube half size: 0.02

evaluate() keys: ['success', 'is_obj_placed', 'is_robot_static', 'is_grasped']
success requires BOTH is_obj_placed AND is_robot_static.


## 2.9.6 — Collected expert episodes

`scripts/generate_expert_demo.py` wraps this recipe into a collector. It wraps
`env.step`, so every transition is recorded as it is executed — nothing is reconstructed
after the fact, and the `(o_t, a_t)` pairing holds by construction.

Recorded result for seeds 0–4:

| Seed | Frames | Success | is_obj_placed | is_robot_static |
|---:|---:|:--:|:--:|:--:|
| 0 | 74 | True | True | True |
| 1 | 74 | True | True | True |
| 2 | 50 | True | True | True |
| 3 | 86 | True | True | True |
| 4 | 76 | True | True | True |

**5 of 5 episodes succeed.** Compare the action smoothness:

| Dataset | mean `|Δa|` |
|---|---|
| random rollout fixture | `0.67` |
| expert planner episodes | `0.0078` |

That is roughly a **100× difference**. Expert actions are small, correlated steps; random
actions are nearly independent draws. This single statistic separates the two regimes, and
it is why the random trajectory can never train a policy no matter how long it trains.

The collector also writes the contract into the HDF5 attributes: `control_mode`,
`data_quality="expert_planner"`, `control_freq_hz=20`,
`timestamp_source="derived_not_measured"`, and the per-channel action semantics.

In [7]:
import h5py
from pathlib import Path

from pathlib import Path

# Notebooks may be started with either the repository root or notebooks/ as the working
# directory. Walk up until the project root is found, so dataset and cache paths never
# resolve to notebooks/datasets or notebooks/.cache by accident.
_cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in (_cwd, *_cwd.parents) if (p / ".git").exists()),
    _cwd.parent if _cwd.name == "notebooks" else _cwd,
)
EXPERT_H5 = PROJECT_ROOT / "datasets" / "pickcube" / "expert_episodes.h5"

if not EXPERT_H5.exists():
    print(f"{EXPERT_H5} not found.")
    print("Generate it with: python scripts/generate_expert_demo.py --seeds 0 1 2 3 4 --overwrite")
else:
    with h5py.File(EXPERT_H5, "r") as handle:
        print("root attributes:")
        for key in ("env_id", "control_mode", "data_quality", "control_freq_hz", "timestamp_source"):
            print(f"  {key:<18} {handle.attrs[key]}")
        print()
        print(f"{'episode':<16} {'frames':>6} {'success':>8} {'|da| mean':>10} {'return':>8}")
        print("-" * 54)
        for name in handle:
            group = handle[name]
            actions = group["actions"][:]
            rewards = group["rewards"][:]
            smoothness = np.abs(np.diff(actions, axis=0)).mean()
            print(f"{name:<16} {actions.shape[0]:>6} {str(bool(group.attrs['success'])):>8} "
                  f"{smoothness:>10.4f} {rewards.sum():>8.3f}")

root attributes:
  env_id             PickCube-v1
  control_mode       pd_joint_pos
  data_quality       expert_planner
  control_freq_hz    20
  timestamp_source   derived_not_measured

episode          frames  success  |da| mean   return
------------------------------------------------------
episode_000000       74     True     0.0078   24.971
episode_000001       74     True     0.0074   23.108
episode_000002       50     True     0.0080   14.799
episode_000003       86     True     0.0077   27.869
episode_000004       76     True     0.0077   25.684


## 2.9.7 — Replay verification, and one honest caveat

An expert dataset is not accepted until replaying it reproduces the recorded outcome. The
collector's episodes were replayed from the same seeds and checked.

| Check | Result |
|---|---|
| Reward per step | **exact** (max error `0.00e+00`) |
| `success` after replay | identical to recorded (5/5) |
| `goal_pos` | exact |
| Continuous state (`qpos`, `qvel`, `tcp_pose`, `obj_pose`) | agrees to `~1e-2` |
| `is_grasped` | differs on **1 frame out of 74**, at the grasp transition |

The `is_grasped` mismatch is worth understanding rather than hiding. It is a **boolean
contact test**, and the frame at which it flips depends on the exact contact solver state:
recorded episode 0 flips at frame 36, replay at frame 37. The physics, the actions, and
every reward are identical.

Lesson: the project's own gate — "replay must match the source exactly" — was
*writable* for the random trajectory because its actions were tiny and deterministic. For
planner-driven expert episodes, exact **observation** equality is not achievable across a
boolean contact threshold; exact **reward** equality and identical **success** are. State
the tolerance you verified instead of claiming bit-exactness.

## 2.9.8 — How this relates to the existing fixture

The repository now holds two kinds of trajectory, and they are **not** interchangeable:

| | `random_episode_standard.h5` | `expert_episodes.h5` |
|---|---|---|
| Source | random actions | motion planner |
| Episodes / frames | 1 / 50 | 5 / 50–86 |
| Success | none (`success_any=False`) | 5/5 |
| Control mode | `pd_joint_delta_pos` | `pd_joint_pos` |
| Action dim | 8 (7 delta + 1 absolute) | 8 (8 absolute) |
| Role | pipeline fixture | imitation-learning supervision |

**The action semantics differ**, even though both are 8-dimensional and both live in a
box. `pd_joint_delta_pos` gives arm **deltas** in `[-0.1, 0.1]` rad plus an absolute
gripper target; `pd_joint_pos` gives **absolute** joint position targets within the joint
limits. Concatenating them without conversion would train a policy on two incompatible
action meanings — the exact failure mode `1.2_action_space_and_control_modes.ipynb`
warns about.

A converter from absolute targets back to deltas is straightforward
(`Δq_t = q_target[t] - qpos[t]`, then scale by `0.1` and clamp to `[-1,1]`), but it must be
built and verified before the two datasets are mixed. That is deliberately **not** done
here.

## Takeaways

1. An expert demonstration comes from a **planner**, not a policy. Random rollouts are
   pipeline fixtures; planners produce supervision.
2. `mplib` 0.1.1 requires `numpy<2`. Under NumPy 2.x it segfaults at address `0x0` because
   its C-API function pointers are invalid. This is an ABI problem, not a GPU problem.
3. The planner requires `pd_joint_pos`; its gripper helpers emit absolute position actions.
4. Four steps succeed: reach → grasp → close → carry. Adding `open_gripper()` or extra
   settling steps breaks placement.
5. Success is `is_obj_placed AND is_robot_static`. Grasping alone is not success.
6. Expert episodes are ~100× smoother than random ones (`|Δa|` 0.0078 vs 0.67).
7. Replay reproduces rewards exactly and success identically; a boolean contact flag can
   still differ on a single frame. Claim the tolerance you actually verified.
8. Expert data in `pd_joint_pos` is **not** directly concatenable with the existing
   `pd_joint_delta_pos` fixture. Convert and verify first.